# Databricks Table Recovery & Storage Notes

## Managed Table Recovery

### Drop Table

```sql
DROP TABLE catalog.schema.table_name;
```

### View Dropped Tables

```sql
SHOW TABLES DROPPED IN schema_name;
```

or

```sql
SHOW TABLES DROPPED IN catalog.schema;
```

### Recover Dropped Table

```sql
UNDROP TABLE table_name;
```

or

```sql
UNDROP TABLE catalog.schema.table_name;
```

### Recover Using Table ID

Useful when a table with the same name already exists.

```sql
UNDROP TABLE WITH ID '<table-id>';
```

Example:

```sql
UNDROP TABLE WITH ID 'absdhhdsjkh';
```

### Retention

* Managed tables can typically be recovered within the retention period (commonly 7 days).
* Metadata and underlying data are retained during this period.
* After retention expires, recovery may not be possible.

---

# External Tables

### Drop External Table

```sql
DROP TABLE catalog.schema.table_name;
```

### What Happens?

* Only table metadata is deleted.
* Physical data files remain in ADLS/S3/GCS.

Example:

```text
abfss://container@storageaccount.dfs.core.windows.net/path/
```

### Recovery

UNDROP is generally not used for external tables.

Recreate the table:

```sql
CREATE TABLE catalog.schema.table_name
USING DELTA
LOCATION 'abfss://container@storageaccount.dfs.core.windows.net/path/';
```

### Important Interview Point

External table deletion removes metadata only.

Data remains in storage unless files are explicitly deleted.

---

# Delta Lake Recovery

## View Table History

```sql
DESCRIBE HISTORY catalog.schema.table_name;
```

Shows:

* INSERT
* UPDATE
* DELETE
* MERGE
* OPTIMIZE
* RESTORE

---

## Query Previous Version (Time Travel)

```sql
SELECT *
FROM catalog.schema.table_name VERSION AS OF 10;
```

or

```sql
SELECT *
FROM catalog.schema.table_name
TIMESTAMP AS OF '2026-06-17 10:00:00';
```

---

## Restore Entire Table

```sql
RESTORE TABLE catalog.schema.table_name
TO VERSION AS OF 10;
```

or

```sql
RESTORE TABLE catalog.schema.table_name
TO TIMESTAMP AS OF '2026-06-17 10:00:00';
```

---

# DROP vs DELETE

## DELETE

```sql
DELETE FROM table_name
WHERE condition;
```

* Deletes records only.
* Table structure remains.
* Recoverable using Time Travel or RESTORE.

## DROP TABLE

```sql
DROP TABLE table_name;
```

* Deletes table object.
* Managed tables can be recovered using UNDROP.
* External tables require recreation from storage location.

---

# Managed vs External Tables

| Feature            | Managed Table                | External Table                |
| ------------------ | ---------------------------- | ----------------------------- |
| Storage Managed By | Databricks                   | Customer                      |
| DROP TABLE         | Metadata + data soft deleted | Metadata removed              |
| UNDROP Supported   | Yes                          | Generally No                  |
| Data Files Remain  | During retention period      | Yes                           |
| Recovery Method    | UNDROP                       | Recreate table using LOCATION |

---

# Storage Architecture Notes

## Databricks Metastore Storage

Example:

```text
abfss://unity-catalog-storage@dbstorage.../root/metastore
```

Purpose:

* Internal Databricks storage
* Managed tables
* Unity Catalog system storage
* Default catalog storage

Not intended for business data organization.

---

## Customer Controlled Storage

Example:

```text
abfss://root@kharvistorage.dfs.core.windows.net/
```

Recommended structure:

```text
bronze/
silver/
gold/
```

Used for:

* External tables
* Lakehouse architecture
* Data sharing
* Storage governance

---

# External Locations

Example:

```sql
CREATE EXTERNAL LOCATION bronze_loc
URL 'abfss://root@kharvistorage.dfs.core.windows.net/bronze'
WITH (STORAGE CREDENTIAL adls_cred);
```

Purpose:

* Secure access to storage paths.
* Control READ FILES and WRITE FILES permissions.
* Does not automatically become catalog storage.

---

# Catalog Storage Behavior

## Catalog Without Location

```sql
CREATE CATALOG sales;
```

Uses:

* Metastore default storage root.

Example:

```text
abfss://unity-catalog-storage@dbstorage.../root/metastore
```

---

## Catalog With Managed Location

```sql
CREATE CATALOG sales
MANAGED LOCATION 'abfss://root@kharvistorage.dfs.core.windows.net/sales';
```

Uses:

* Customer-managed storage.

Recommended for production environments.

---

# Important Unity Catalog Concepts

## External Location

* Approved storage path.
* Provides secure access.
* Does not store metadata.

## Catalog

* Top-level container.
* Holds schemas and tables.

## Schema

* Database inside catalog.
* Holds tables and views.

## Metastore

* Stores metadata and permissions.
* Does not store actual business data.

---

# Useful Commands

## Show Catalog Details

```sql
DESCRIBE CATALOG EXTENDED catalog_name;
```

## Show Schema Details

```sql
DESCRIBE SCHEMA EXTENDED catalog.schema;
```

## Show Table Details

```sql
DESCRIBE DETAIL catalog.schema.table_name;
```

## View Storage Location

```sql
DESCRIBE DETAIL catalog.schema.table_name;
```

Look for:

```text
location
```

## View History

```sql
DESCRIBE HISTORY catalog.schema.table_name;
```

## Restore Table

```sql
RESTORE TABLE catalog.schema.table_name
TO VERSION AS OF <version>;
```

## Recover Managed Table

```sql
UNDROP TABLE catalog.schema.table_name;
```

## Recover Using Table ID

```sql
UNDROP TABLE WITH ID '<table-id>';
```

---

# Best Practices

1. Use Delta format for all production tables.
2. Use External Locations for bronze/silver/gold layers.
3. Use customer-managed storage for critical data.
4. Always know the physical storage location of important tables.
5. Use Time Travel and RESTORE before considering data reloads.
6. Keep managed and external table usage clearly documented.
7. Use MANAGED LOCATION when creating production catalogs.









